In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pprint
from scipy.stats import chi2_contingency, ttest_ind, pointbiserialr
from IPython.display import display, HTML
from pathlib import Path

import os
import sys
import src.analysis_utility_funcs as unified_funcs # ---- this worked because of the new pyproject.toml file, and we ran 'pip install -e .' in the conda env setup script, which runs the pyproject.toml, which basically makes any (root-level) cross-folder imports work without needing to mess with sys.path. So I can import python scripts if I want from src, run_annotation, archive, whichever.

## This notebook is where :

1. ## we attach human ratings and original scenario texts to our annotator results (to make a `compact_df`) and 
2. ## use it to do the primary harm picking (which makes a `primary_harm_df`). 

## And we save both as CSVs to be loaded in to the Franken analysis notebook whenever needed. (I.E. for Experiments 1A, 1B, 5A, and 5B).

In [2]:
all_FRANKEN_df = []

intensities = ["mild_harm_mild_good", "severe_harm_very_good"]
causal_conditions = ["cc_evitable_action_yes_stories", "cc_evitable_prevention_no_stories", "cc_inevitable_action_yes_stories", "cc_inevitable_prevention_no_stories", "coc_evitable_action_yes_stories", "coc_evitable_prevention_no_stories", "coc_inevitable_action_yes_stories", "coc_inevitable_prevention_no_stories"]

for intensity in intensities:
    for causal_condition in causal_conditions:
        df = unified_funcs.read_all_scenarios(f"../../scenarios_inputs/franken/conditions_{intensity}/{causal_condition}.json", f"../../annotated_outputs/franken_o1o2/conditions_{intensity}/{causal_condition}/")
        all_FRANKEN_df.append(df)
        # SPECIAL - add the intensity and causal_condition as columns to the dataframe
        all_FRANKEN_df[-1]["intensity"] = intensity
        all_FRANKEN_df[-1]["causal_condition"] = causal_condition
all_FRANKEN_df = pd.concat(all_FRANKEN_df, ignore_index=True)
# sort the intensity column by the order of the intensities in the intensities list above
all_FRANKEN_df["intensity"] = pd.Categorical(all_FRANKEN_df["intensity"], categories=intensities, ordered=True)
# sort them by the order of the causal conditions in the causal_conditions list above
all_FRANKEN_df["causal_condition"] = pd.Categorical(all_FRANKEN_df["causal_condition"], categories=causal_conditions, ordered=True)
# sort the dataframe by intensity, then by causal condition, then by SID, then by option --- this is how I think is best to sort it for readability but can be changed up.
all_FRANKEN_df = all_FRANKEN_df.sort_values(by=["intensity","causal_condition", "SID", "option"]).reset_index(drop=True)
# drop all the rows where SID is greater than 9
all_FRANKEN_df = all_FRANKEN_df[all_FRANKEN_df["SID"] <= 9]
display(HTML(all_FRANKEN_df[0:10].to_html()))
print("number of rows in all_FRANKEN_df:", len(all_FRANKEN_df))

,SID,option,event,being,C,I,K,utility,intensity,causal_condition
0,0,1: renovate the park.,The park closes temporarily,i,+,+,+,"{'i': '0', 'residents of the community who use or value the park': '-55'}",mild_harm_mild_good,cc_evitable_action_yes_stories
1,0,1: renovate the park.,Residents lose access to the recreational space during the renovation,i,+,+,+,"{'i': '0', 'residents of the community who use or value the park': '-68'}",mild_harm_mild_good,cc_evitable_action_yes_stories
2,0,1: renovate the park.,Residents experience inconvenience,i,+,-,+,"{'i': '0', 'residents of the community who use or value the park': '-35'}",mild_harm_mild_good,cc_evitable_action_yes_stories
3,0,1: renovate the park.,Residents experience disappointment,i,+,-,+,"{'i': '-12', 'residents of the community who use or value the park': '-55'}",mild_harm_mild_good,cc_evitable_action_yes_stories
4,0,1: renovate the park.,Park facilities are improved,i,+,+,+,"{'i': '12', 'residents of the community who use or value the park': '84'}",mild_harm_mild_good,cc_evitable_action_yes_stories
5,1,1: upgrade the server system.,I take my company's server system offline for a few hours,i,+,+,+,{'i': '-35'},mild_harm_mild_good,cc_evitable_action_yes_stories
6,1,1: upgrade the server system.,The company's operations temporarily halt,i,+,-,+,{'i': '-35'},mild_harm_mild_good,cc_evitable_action_yes_stories
7,1,1: upgrade the server system.,Data processing speed slightly improves after the upgrade,i,+,+,+,{'i': '12'},mild_harm_mild_good,cc_evitable_action_yes_stories
8,1,1: upgrade the server system.,Customers or users are unable to access server-dependent services during the downtime,i,+,-,+,{'i': '0'},mild_harm_mild_good,cc_evitable_action_yes_stories
9,2,1: implement the new teaching method.,Students attend class one hour earlier,i,+,+,+,"{'students not required to attend class one hour earlier': '0', 'i': '0', 'students required to attend class one hour earlier': '-15'}",mild_harm_mild_good,cc_evitable_action_yes_stories


number of rows in all_FRANKEN_df: 754


In [3]:
mild_FRANKEN_df = all_FRANKEN_df[all_FRANKEN_df["intensity"] == "mild_harm_mild_good"]
severe_FRANKEN_df = all_FRANKEN_df[all_FRANKEN_df["intensity"] == "severe_harm_very_good"]
print("mild_FRANKEN_df preview:")
display(HTML(mild_FRANKEN_df[0:10].to_html()))
print("number of rows in mild_FRANKEN_df:", len(mild_FRANKEN_df))
print("severe_FRANKEN_df preview:")
display(HTML(severe_FRANKEN_df[0:10].to_html()))
print("number of rows in severe_FRANKEN_df:", len(severe_FRANKEN_df))

mild_FRANKEN_df preview:


,SID,option,event,being,C,I,K,utility,intensity,causal_condition
0,0,1: renovate the park.,The park closes temporarily,i,+,+,+,"{'i': '0', 'residents of the community who use or value the park': '-55'}",mild_harm_mild_good,cc_evitable_action_yes_stories
1,0,1: renovate the park.,Residents lose access to the recreational space during the renovation,i,+,+,+,"{'i': '0', 'residents of the community who use or value the park': '-68'}",mild_harm_mild_good,cc_evitable_action_yes_stories
2,0,1: renovate the park.,Residents experience inconvenience,i,+,-,+,"{'i': '0', 'residents of the community who use or value the park': '-35'}",mild_harm_mild_good,cc_evitable_action_yes_stories
3,0,1: renovate the park.,Residents experience disappointment,i,+,-,+,"{'i': '-12', 'residents of the community who use or value the park': '-55'}",mild_harm_mild_good,cc_evitable_action_yes_stories
4,0,1: renovate the park.,Park facilities are improved,i,+,+,+,"{'i': '12', 'residents of the community who use or value the park': '84'}",mild_harm_mild_good,cc_evitable_action_yes_stories
5,1,1: upgrade the server system.,I take my company's server system offline for a few hours,i,+,+,+,{'i': '-35'},mild_harm_mild_good,cc_evitable_action_yes_stories
6,1,1: upgrade the server system.,The company's operations temporarily halt,i,+,-,+,{'i': '-35'},mild_harm_mild_good,cc_evitable_action_yes_stories
7,1,1: upgrade the server system.,Data processing speed slightly improves after the upgrade,i,+,+,+,{'i': '12'},mild_harm_mild_good,cc_evitable_action_yes_stories
8,1,1: upgrade the server system.,Customers or users are unable to access server-dependent services during the downtime,i,+,-,+,{'i': '0'},mild_harm_mild_good,cc_evitable_action_yes_stories
9,2,1: implement the new teaching method.,Students attend class one hour earlier,i,+,+,+,"{'students not required to attend class one hour earlier': '0', 'i': '0', 'students required to attend class one hour earlier': '-15'}",mild_harm_mild_good,cc_evitable_action_yes_stories


number of rows in mild_FRANKEN_df: 309
severe_FRANKEN_df preview:


,SID,option,event,being,C,I,K,utility,intensity,causal_condition
380,0,1: initiate the major urban renewal project.,Historical buildings are demolished,i,+,+,+,"{'i': '0', 'residents of the long-established residential community': '-72'}",severe_harm_very_good,cc_evitable_action_yes_stories
381,0,1: initiate the major urban renewal project.,Residents are relocated from the long-established community,i,+,+,+,"{'i': '-72', 'residents of the long-established residential community': '-88'}",severe_harm_very_good,cc_evitable_action_yes_stories
382,0,1: initiate the major urban renewal project.,I initiate the major urban renewal project,i,+,+,+,"{'i': '78', 'residents of the long-established residential community': '-72'}",severe_harm_very_good,cc_evitable_action_yes_stories
383,0,1: initiate the major urban renewal project.,A new business district is created,i,+,+,+,"{'i': '0', 'residents of the long-established residential community': '-35'}",severe_harm_very_good,cc_evitable_action_yes_stories
384,0,1: initiate the major urban renewal project.,International investment is attracted to the city,i,+,+,-,"{'i': '12', 'residents of the long-established residential community': '35'}",severe_harm_very_good,cc_evitable_action_yes_stories
385,0,1: initiate the major urban renewal project.,The local economy increases,i,+,+,+,"{'i': '0', 'residents of the long-established residential community': '72'}",severe_harm_very_good,cc_evitable_action_yes_stories
386,0,1: initiate the major urban renewal project.,Job opportunities increase,i,+,+,+,"{'i': '72', 'residents of the long-established residential community': '72'}",severe_harm_very_good,cc_evitable_action_yes_stories
387,0,1: initiate the major urban renewal project.,Residents experience disruption to their homes and community ties,i,+,-,+,"{'i': '-72', 'residents of the long-established residential community': '-88'}",severe_harm_very_good,cc_evitable_action_yes_stories
388,0,1: initiate the major urban renewal project.,People lose access to historically significant buildings,i,+,+,+,"{'i': '0', 'residents of the long-established residential community': '-62'}",severe_harm_very_good,cc_evitable_action_yes_stories
389,1,1: implement the new power grid system.,I implement the new power grid system,i,+,+,+,"{'i': '78', 'city residents': '78'}",severe_harm_very_good,cc_evitable_action_yes_stories


number of rows in severe_FRANKEN_df: 445


### Step 1 - Load the human ratings into a DF (from Franken Exp 2 for intentionality and permissibility)

In [4]:
exp2_data_path = "../../human_data/franken/exp2_moralperm-intent-rating"
exp2_raw_df = []
# go through all json files in the exp2_data_path
for json_file in Path(exp2_data_path).glob("*.json"):
    # print(f"Processing file: {json_file.name}")
    with open(json_file, 'r') as f:
        curr_batch_list = json.load(f)
        for exp2_item in curr_batch_list:
            exp2_raw_df.append(exp2_item)
# reorder the exp2_raw_df by the "folder_name" key according to the folder_order list
condition_order = [
    "cc_evitable_action_yes",
    "cc_evitable_prevention_no",
    "cc_inevitable_action_yes",
    "cc_inevitable_prevention_no",
    "coc_evitable_action_yes",
    "coc_evitable_prevention_no",
    "coc_inevitable_action_yes",
    "coc_inevitable_prevention_no"
]
exp2_raw_df.sort(key=lambda x: condition_order.index(x["condition"]))
# reorder the exp2_raw_df by the "scenario_id" key in ascending order
exp2_raw_df.sort(key=lambda x: int(x["scenario_id"]))
exp2_raw_df = pd.DataFrame(exp2_raw_df)
# print("Raw DataFrame from Experiment 2:")
# display(HTML(exp2_raw_df[0:2].to_html()))
# print(f"Total number of rows in Exp2 raw DataFrame: {len(exp2_raw_df)}")

human_ratings = pd.read_csv(Path().resolve() / "../../human_data/franken/exp2_moralperm-intent-rating/data_long_format.csv")
human_ratings = human_ratings.drop(columns=['scenario_harm', 'split'])
# count number of unique combinations of scenario_id + causal_structure + evitability + action
exp2_ratings_df = human_ratings.groupby(['scenario_id', 'causal_structure', 'evitability', 'action']).size().reset_index(name='counts') # each scenario got rated by ~20-25 participants
# add a column of average rating of moral permissibility and intention for each unique combination
avg_ratings = human_ratings.groupby(['scenario_id', 'causal_structure', 'evitability', 'action'])[['permissibility_rating', 'intention_rating']].mean().reset_index()
exp2_ratings_df = exp2_ratings_df.merge(avg_ratings, on=['scenario_id', 'causal_structure', 'evitability', 'action'])
exp2_ratings_df = exp2_ratings_df.rename(columns={
    'permissibility_rating': 'avg_permissibility_rating',
    'intention_rating': 'avg_intention_rating'
})
print("\n Explainer: \n causal_structure (0 for means (cc), 1 for side effect (coc)) \n action (0 for commission i.e. action_yes, 1 for omission i.e. prevention_no) \n evitability (0 for evitable, 1 for inevitable) \n avg_permissibility_rating (average moral permissibility rating for that scenario) \n avg_intention_rating (average intention rating for that scenario)\n")
print("Processed DataFrame with average ratings from Experiment 2:")
display(HTML(exp2_ratings_df[0:6].to_html()))
print(f"Total number of rows in processed Exp2 ratings DataFrame: {len(exp2_ratings_df)}")


 Explainer: 
 causal_structure (0 for means (cc), 1 for side effect (coc)) 
 action (0 for commission i.e. action_yes, 1 for omission i.e. prevention_no) 
 evitability (0 for evitable, 1 for inevitable) 
 avg_permissibility_rating (average moral permissibility rating for that scenario) 
 avg_intention_rating (average intention rating for that scenario)

Processed DataFrame with average ratings from Experiment 2:


,scenario_id,causal_structure,evitability,action,counts,avg_permissibility_rating,avg_intention_rating
0,0,0,0,0,26,4.153846,1.923077
1,0,0,0,1,25,4.280000,1.960000
2,0,0,1,0,21,3.952381,2.476190
3,0,0,1,1,21,3.952381,2.666667
4,0,1,0,0,22,4.272727,2.045455
5,0,1,0,1,21,4.238095,1.952381


Total number of rows in processed Exp2 ratings DataFrame: 80


### Step 2 - Add the human ratings to our mild_FRANKEN_df (ratings only exist for mild scenarios)

In [5]:
# create a new dataframe to hold the merged data
mild_FRANKEN_withhuman_df = mild_FRANKEN_df.copy()
# create new columns for avg_permissibility_rating and avg_intention_rating in the merged_df and initialize them with NaN
mild_FRANKEN_withhuman_df["avg_permissibility_rating"] = np.nan
mild_FRANKEN_withhuman_df["avg_intention_rating"] = np.nan  

# loop through each row in the merged_df and find the corresponding avg ratings from exp2_ratings_df
for idx, row in mild_FRANKEN_withhuman_df.iterrows():
    sid = row["SID"]
    causal_condition = row["causal_condition"]

    # determine the values of causal_structure, evitability, and action based on the causal_condition string
    if "cc" in causal_condition:
        causal_structure = 0
    elif "coc" in causal_condition:
        causal_structure = 1
    else:
        continue  # skip if neither is found

    if "evitable" in causal_condition:
        evitability = 0
    elif "inevitable" in causal_condition:
        evitability = 1
    else:
        continue  # skip if neither is found

    if "action_yes" in causal_condition:
        action = 0
    elif "prevention_no" in causal_condition:
        action = 1
    else:
        continue  # skip if neither is found

    # find the matching row in exp2_ratings_df
    match = exp2_ratings_df[
        (exp2_ratings_df["scenario_id"] == sid) &
        (exp2_ratings_df["causal_structure"] == causal_structure) &
        (exp2_ratings_df["evitability"] == evitability) &
        (exp2_ratings_df["action"] == action)
    ]

    if not match.empty:
        mild_FRANKEN_withhuman_df.at[idx, "avg_permissibility_rating"] = match.iloc[0]["avg_permissibility_rating"]
        mild_FRANKEN_withhuman_df.at[idx, "avg_intention_rating"] = match.iloc[0]["avg_intention_rating"]

print("Our old mild_FRANKEN_df with average human ratings per scenario added from Experiment 2:")
display(HTML(mild_FRANKEN_withhuman_df[0:10].to_html()))
print(f"Total number of rows in mild_FRANKEN_withhuman_df: {len(mild_FRANKEN_withhuman_df)}")

Our old mild_FRANKEN_df with average human ratings per scenario added from Experiment 2:


,SID,option,event,being,C,I,K,utility,intensity,causal_condition,avg_permissibility_rating,avg_intention_rating
0,0,1: renovate the park.,The park closes temporarily,i,+,+,+,"{'i': '0', 'residents of the community who use or value the park': '-55'}",mild_harm_mild_good,cc_evitable_action_yes_stories,4.153846,1.923077
1,0,1: renovate the park.,Residents lose access to the recreational space during the renovation,i,+,+,+,"{'i': '0', 'residents of the community who use or value the park': '-68'}",mild_harm_mild_good,cc_evitable_action_yes_stories,4.153846,1.923077
2,0,1: renovate the park.,Residents experience inconvenience,i,+,-,+,"{'i': '0', 'residents of the community who use or value the park': '-35'}",mild_harm_mild_good,cc_evitable_action_yes_stories,4.153846,1.923077
3,0,1: renovate the park.,Residents experience disappointment,i,+,-,+,"{'i': '-12', 'residents of the community who use or value the park': '-55'}",mild_harm_mild_good,cc_evitable_action_yes_stories,4.153846,1.923077
4,0,1: renovate the park.,Park facilities are improved,i,+,+,+,"{'i': '12', 'residents of the community who use or value the park': '84'}",mild_harm_mild_good,cc_evitable_action_yes_stories,4.153846,1.923077
5,1,1: upgrade the server system.,I take my company's server system offline for a few hours,i,+,+,+,{'i': '-35'},mild_harm_mild_good,cc_evitable_action_yes_stories,4.095238,2.095238
6,1,1: upgrade the server system.,The company's operations temporarily halt,i,+,-,+,{'i': '-35'},mild_harm_mild_good,cc_evitable_action_yes_stories,4.095238,2.095238
7,1,1: upgrade the server system.,Data processing speed slightly improves after the upgrade,i,+,+,+,{'i': '12'},mild_harm_mild_good,cc_evitable_action_yes_stories,4.095238,2.095238
8,1,1: upgrade the server system.,Customers or users are unable to access server-dependent services during the downtime,i,+,-,+,{'i': '0'},mild_harm_mild_good,cc_evitable_action_yes_stories,4.095238,2.095238
9,2,1: implement the new teaching method.,Students attend class one hour earlier,i,+,+,+,"{'students not required to attend class one hour earlier': '0', 'i': '0', 'students required to attend class one hour earlier': '-15'}",mild_harm_mild_good,cc_evitable_action_yes_stories,4.047619,2.095238


Total number of rows in mild_FRANKEN_withhuman_df: 309


### Step 3 -- "compact" the DF i.e. collect all events into one row per scenario, also add in the full scenario text

In [6]:
# create a compact version of the mild_FRANKEN_withhuman_df by collecting all the rows per scenario into a one row. You can find all the rows that belong to a scenario by finding all the rows that have the same SID and causal condition. The df will have all the rows that are in mild_FRANKEN_withhuman_df, but instead of 'event', we will have 'events' where values are a list of all the event strings. Then, values in the C, I, K columns will also be lists of the corresponding C, I, K values for each event, and the utility column will contain a list of dictionaries, one dict for each event. The intensity, causal_condition, avg_permissibility_rating, and avg_intention_rating columns will just take the value from the first row of the scenario group since they are the same across all rows of the same scenario.

mild_FRANKEN_compact_df = mild_FRANKEN_withhuman_df.groupby(["SID", "causal_condition", "option", "being", "intensity", "avg_permissibility_rating", "avg_intention_rating"]).agg({
    "event": list,
    "C": list,
    "I": list,
    "K": list,
    "utility": list
}).reset_index()
# reorder the order of the columns to be SID, intensity, causal_condition, option, events, being, C, I, K, utility, avg_permissibility_rating, avg_intention_rating
mild_FRANKEN_compact_df = mild_FRANKEN_compact_df[["SID", "intensity", "causal_condition", "option", "event", "being", "C", "I", "K", "utility", "avg_permissibility_rating", "avg_intention_rating"]]
# rename the "event" column to "events" since it now contains a list of events
mild_FRANKEN_compact_df = mild_FRANKEN_compact_df.rename(columns={"event": "events"})


# last thing -- bring in the original scenario text into the mild_FRANKEN_compact_df for easier reference. We need to get them from the JSONs in ../../scenarios_inputs/franken/conditions_mild_harm_mild_good/ where we have eight JSON files corresponding to the eight causal conditions (files have the same names as the causal conditions in the mild_FRANKEN_compact_df). 
scenario_texts = {}
for causal_condition in causal_conditions:
    with open(f"../../scenarios_inputs/franken/conditions_mild_harm_mild_good/{causal_condition}.json", 'r') as f:
        scenarios = json.load(f)
        for scenario in scenarios:
            scenario_texts[(scenario["id"], causal_condition)] = scenario["text"]
# now create a new column in mild_FRANKEN_compact_df called "scenario_text" and fill it by matching the SID and causal_condition with the keys in scenario_texts
mild_FRANKEN_compact_df["scenario_text"] = mild_FRANKEN_compact_df.apply(lambda row: scenario_texts.get((row["SID"], row["causal_condition"]), "N/A"), axis=1)
# reorder the columns to put scenario_text right after causal_condition for easier reference
mild_FRANKEN_compact_df = mild_FRANKEN_compact_df[["SID", "intensity", "causal_condition", "scenario_text", "option", "events", "being", "C", "I", "K", "utility", "avg_permissibility_rating", "avg_intention_rating"]]
# group the rows by causal condition and then SID and thats it:
mild_FRANKEN_compact_df = mild_FRANKEN_compact_df.groupby(["causal_condition", "SID"]).first().reset_index()
print("Mild_FRANKEN_compact_df with scenario_text column added:")
display(HTML(mild_FRANKEN_compact_df[0:2].to_html()))
print(f"Total number of rows in mild_FRANKEN_compact_df: {len(mild_FRANKEN_compact_df)}")

# save this df into a csv file for easier reference in the future
mild_FRANKEN_compact_df.to_csv("mild_FRANKEN_compact_df.csv", index=False)

Mild_FRANKEN_compact_df with scenario_text column added:


,causal_condition,SID,intensity,scenario_text,option,events,being,C,I,K,utility,avg_permissibility_rating,avg_intention_rating
0,cc_evitable_action_yes_stories,0,mild_harm_mild_good,"I, an Architect, face a moral dilemma. I have the opportunity to renovate a park in the city. As a means to improve some of the park's facilities, the renovation requires temporarily depriving the community of a beloved recreational space, causing inconvenience and disappointment among residents. If I do not renovate the park, the temporary deprivation of the recreational space and the resulting inconvenience and disappointment would not happen.",1: renovate the park.,"[The park closes temporarily, Residents lose access to the recreational space during the renovation, Residents experience inconvenience, Residents experience disappointment, Park facilities are improved]",i,"[+, +, +, +, +]","[+, +, -, -, +]","[+, +, +, +, +]","[{'i': '0', 'residents of the community who use or value the park': '-55'}, {'i': '0', 'residents of the community who use or value the park': '-68'}, {'i': '0', 'residents of the community who use or value the park': '-35'}, {'i': '-12', 'residents of the community who use or value the park': '-55'}, {'i': '12', 'residents of the community who use or value the park': '84'}]",4.153846,1.923077
1,cc_evitable_action_yes_stories,1,mild_harm_mild_good,"I, an Engineer, face a moral dilemma. I have the opportunity to upgrade the company's server system. As a means to slightly improve data processing speed, the upgrade process requires the company's server system to be offline for a few hours, causing a temporary halt in the company's operations. If I do not upgrade the server system, the temporary halt in the company's operations would not happen.",1: upgrade the server system.,"[I take my company's server system offline for a few hours, The company's operations temporarily halt, Data processing speed slightly improves after the upgrade, Customers or users are unable to access server-dependent services during the downtime]",i,"[+, +, +, +]","[+, -, +, -]","[+, +, +, +]","[{'i': '-35'}, {'i': '-35'}, {'i': '12'}, {'i': '0'}]",4.095238,2.095238


Total number of rows in mild_FRANKEN_compact_df: 80


### Did the harm-picking from the DF above, index of picked event for each scenario is in `mild_FRANKEN_harm_picks.csv`

### Making a "primary harm picks DF":

In [7]:
# Use the mild_FRANKEN_harm_picks.csv to make a "primary harm DF". Read in "mild_FRANKEN_harm_picks.csv", which just contains an integer in every row. Loop through mild_FRANKEN_compact_df, and for each row, look at the "event" column, which is a list of event strings. Use the integer in the corresponding row of mild_FRANKEN_harm_picks.csv to pick out one of the events from the "event" list. Then create a new dataframe called mild_FRANKEN_primary_harm_df, which has the same columns as mild_FRANKEN_compact_df, but the "event" column is now just the single event string that was picked out using the integer from mild_FRANKEN_harm_picks.csv.

harm_picks_df = pd.read_csv("mild_FRANKEN_harm_picks.csv")
mild_FRANKEN_primary_harm_df = mild_FRANKEN_compact_df.copy()
mild_FRANKEN_primary_harm_df["picked_event"] = None
for idx, row in mild_FRANKEN_primary_harm_df.iterrows():
    pick_idx = harm_picks_df.iloc[idx]["pick"]
    mild_FRANKEN_primary_harm_df.at[idx, "picked_event"] = row["events"][pick_idx]
    mild_FRANKEN_primary_harm_df.at[idx, "picked_event_idx"] = int(pick_idx)
    mild_FRANKEN_primary_harm_df.at[idx, "C"] = row["C"][pick_idx]
    mild_FRANKEN_primary_harm_df.at[idx, "I"] = row["I"][pick_idx]
    mild_FRANKEN_primary_harm_df.at[idx, "K"] = row["K"][pick_idx]
    mild_FRANKEN_primary_harm_df.at[idx, "utility"] = row["utility"][pick_idx]

# reorder the columns to put picked_event right after scenario_text for easier reference
mild_FRANKEN_primary_harm_df = mild_FRANKEN_primary_harm_df[["SID", "intensity", "causal_condition", "scenario_text", "events", "picked_event", "picked_event_idx", "option", "being", "C", "I", "K", "utility", "avg_permissibility_rating", "avg_intention_rating"]]
print("Mild_FRANKEN_primary_harm_df:")
display(HTML(mild_FRANKEN_primary_harm_df[0:2].to_html()))
print(f"Total number of rows in mild_FRANKEN_primary_harm_df: {len(mild_FRANKEN_primary_harm_df)}")

# save this as mild_FRANKEN_primary_harm_df.csv for easier reference in the future
mild_FRANKEN_primary_harm_df.to_csv("mild_FRANKEN_primary_harm_df.csv", index=False)


Mild_FRANKEN_primary_harm_df:


,SID,intensity,causal_condition,scenario_text,events,picked_event,picked_event_idx,option,being,C,I,K,utility,avg_permissibility_rating,avg_intention_rating
0,0,mild_harm_mild_good,cc_evitable_action_yes_stories,"I, an Architect, face a moral dilemma. I have the opportunity to renovate a park in the city. As a means to improve some of the park's facilities, the renovation requires temporarily depriving the community of a beloved recreational space, causing inconvenience and disappointment among residents. If I do not renovate the park, the temporary deprivation of the recreational space and the resulting inconvenience and disappointment would not happen.","[The park closes temporarily, Residents lose access to the recreational space during the renovation, Residents experience inconvenience, Residents experience disappointment, Park facilities are improved]",Residents lose access to the recreational space during the renovation,1.0,1: renovate the park.,i,+,+,+,"{'i': '0', 'residents of the community who use or value the park': '-68'}",4.153846,1.923077
1,1,mild_harm_mild_good,cc_evitable_action_yes_stories,"I, an Engineer, face a moral dilemma. I have the opportunity to upgrade the company's server system. As a means to slightly improve data processing speed, the upgrade process requires the company's server system to be offline for a few hours, causing a temporary halt in the company's operations. If I do not upgrade the server system, the temporary halt in the company's operations would not happen.","[I take my company's server system offline for a few hours, The company's operations temporarily halt, Data processing speed slightly improves after the upgrade, Customers or users are unable to access server-dependent services during the downtime]",The company's operations temporarily halt,1.0,1: upgrade the server system.,i,+,-,+,{'i': '-35'},4.095238,2.095238


Total number of rows in mild_FRANKEN_primary_harm_df: 80


### Expand the DF above if you want to see what primary harm I picked for each of the 80 scenarios!
### (Can drop some columns and re-print it for less visual crowding if you need!)